In [ ]:
# ── Setup: LLM, Tools, Agent, and Logger ──

import os, json, base64, shutil, re, requests
from datetime import datetime
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

OUTPUT_DIR = "output"
DIAGRAMS_DIR = f"{OUTPUT_DIR}/diagrams"
SECTIONS_FILE = f"{OUTPUT_DIR}/sections.json"
DIAGRAM_META_FILE = f"{OUTPUT_DIR}/diagram_meta.json"
LOG_FILE = f"{OUTPUT_DIR}/agent_log.txt"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)


# ── Helpers ──

def load_json(path):
    """Load a JSON list from file, returning [] if missing."""
    try:
        with open(path) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return []


def save_json(path, data):
    """Write data as JSON to file."""
    with open(path, "w") as f:
        json.dump(data, f)


def log(msg: str):
    """Print and persist a timestamped log line."""
    ts = datetime.now().strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")


# ── Logger Callback ──

class AgentLogger(BaseCallbackHandler):
    """Logs every decision, action, and result from the agent loop."""

    def on_agent_action(self, action, **kwargs):
        log(f"DECIDING  -> Use tool: {action.tool}")
        log(f"INPUT     -> {str(action.tool_input)[:200]}")

    def on_tool_end(self, output, **kwargs):
        log(f"RESULT    -> {str(output)[:200]}")

    def on_agent_finish(self, finish, **kwargs):
        log(f"DONE      -> Agent reached final answer ({len(finish.return_values.get('output', ''))} chars)")

    def on_llm_start(self, serialized, prompts, **kwargs):
        log("THINKING  -> LLM reasoning...")

    def on_llm_end(self, response, **kwargs):
        log("THOUGHT   -> LLM produced response")


logger = AgentLogger()


# ── Tools ──

@tool
def create_mermaid_diagram(mermaid_code: str, label: str = "") -> str:
    """Render Mermaid syntax to a PNG file. Pass raw Mermaid code only (no markdown fences).
    Optionally pass a label describing what the diagram shows (used in PDF).
    Use semicolons or newlines to separate statements. ASCII only in labels."""
    log("MERMAID   -> Rendering diagram via mermaid.ink...")

    code = mermaid_code.strip()
    if code.startswith("```"):
        code = re.sub(r"^```(?:mermaid)?\s*\n?", "", code)
        code = re.sub(r"\n?```\s*$", "", code)

    # mermaid.ink requires real newlines, LLMs often use semicolons
    code = code.replace("; ", "\n").replace(";", "\n")

    encoded = base64.urlsafe_b64encode(code.encode()).decode()
    resp = requests.get(f"https://mermaid.ink/img/{encoded}?type=png&bgColor=white", timeout=30)
    if resp.status_code != 200:
        log(f"MERMAID   -> FAILED (HTTP {resp.status_code})")
        return f"Error: HTTP {resp.status_code}. Check syntax. Code was:\n{code[:200]}"

    idx = len([f for f in os.listdir(DIAGRAMS_DIR) if f.endswith(".png")]) + 1
    path = f"{DIAGRAMS_DIR}/diagram_{idx}.png"
    with open(path, "wb") as f:
        f.write(resp.content)

    meta = load_json(DIAGRAM_META_FILE)
    meta.append({
        "path": path,
        "label": label or f"Diagram {idx}",
        "after_section": len(load_json(SECTIONS_FILE)),
    })
    save_json(DIAGRAM_META_FILE, meta)

    log(f"MERMAID   -> Saved {path} ({len(resp.content)} bytes)")
    return f"Diagram saved to {path}"


@tool
def save_research_section(title: str, content: str) -> str:
    """Save a research section (title + content) to be included in the final PDF report."""
    sections = load_json(SECTIONS_FILE)
    sections.append({"title": title, "content": content})
    save_json(SECTIONS_FILE, sections)
    log(f"SAVED     -> Section '{title}' ({len(content)} chars, total: {len(sections)})")
    return f"Saved '{title}' ({len(content)} chars). Total: {len(sections)}"


tools = [
    DuckDuckGoSearchRun(),
    WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)),
    create_mermaid_diagram,
    save_research_section,
]
log("SETUP     -> Tools ready: " + ", ".join(t.name for t in tools))

In [ ]:
# ── Run Agent ──

# Clean previous output and initialize directories
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(DIAGRAMS_DIR, exist_ok=True)

# Tool-calling agent (uses native function calling — no stop sequences needed)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research agent. Use the provided tools to research topics, save findings, and create visual diagrams."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True, max_iterations=25, callbacks=[logger])

# Get user input
user_topic = input("Enter your research topic: ").strip()
if not user_topic:
    raise ValueError("Topic cannot be empty")
log(f"START     -> Topic: {user_topic}")

SYSTEM_PROMPT = f"""Research the following topic: {user_topic}

You MUST complete ALL of these steps:

1. RESEARCH using duckduckgo_search and wikipedia to gather detailed information.

2. SAVE each finding using save_research_section with a title and content.

3. CREATE a Mermaid diagram IMMEDIATELY AFTER each related section using create_mermaid_diagram.
   - Pass a descriptive "label" parameter so the diagram is captioned in the PDF.
   - Use semicolons to separate statements. ASCII only. No markdown fences.
   - Valid example: graph TD; A[Start] --> B[Step 1]; B --> C[Step 2]; C --> D[End]
   - Create at least 2-3 diagrams total, placed after the sections they illustrate.

Complete ALL steps before giving your final answer."""

log("AGENT     -> Execution started")
response = executor.invoke({"input": SYSTEM_PROMPT})
log("AGENT     -> Execution complete")
log(f"OUTPUT    -> {len(response['output'])} chars")
print(f"\nFinal Output:\n{response['output']}")

In [ ]:
# ── Generate PDF Report ──

from fpdf import FPDF
from glob import glob
from PIL import Image

PAGE_W, PAGE_H, MARGIN = 210, 297, 20
MAX_IMG_H = PAGE_H - MARGIN - 22 - 24

UNICODE_TO_ASCII = {
    '\u2014': '-', '\u2013': '-',           # em/en dash
    '\u201c': '"', '\u201d': '"',           # curly double quotes
    '\u2018': "'", '\u2019': "'",           # curly single quotes
    '\u2022': '*', '\u2026': '...',         # bullet, ellipsis
}


def sanitize(text: str) -> str:
    """Replace Unicode chars with ASCII equivalents for FPDF."""
    for old, new in UNICODE_TO_ASCII.items():
        text = text.replace(old, new)
    return text


class ResearchPDF(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(100, 100, 100)
        self.cell(0, 8, "Research Report", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f"Page {self.page_no()}/{{nb}}", align="C")

    def write_markdown(self, text):
        """Render text with **bold** support."""
        text = sanitize(text)
        self.set_font("Helvetica", "", 11)
        self.set_text_color(50, 50, 50)
        for part in re.split(r"(\*\*.*?\*\*)", text):
            if part.startswith("**") and part.endswith("**"):
                self.set_font("Helvetica", "B", 11)
                self.write(6, part[2:-2])
                self.set_font("Helvetica", "", 11)
            else:
                self.write(6, part)

    def add_image_fit(self, img_path, label):
        """Add image scaled to fit page, with label and auto page break."""
        with Image.open(img_path) as img:
            w_px, h_px = img.size
        aspect = w_px / h_px
        img_w, img_h = 160, 160 / aspect
        if img_h > MAX_IMG_H:
            img_h = MAX_IMG_H
            img_w = img_h * aspect
        if 28 + img_h > PAGE_H - MARGIN - self.get_y():
            self.add_page()
        self.ln(4)
        self.set_font("Helvetica", "BI", 11)
        self.set_text_color(60, 60, 60)
        self.cell(0, 8, sanitize(label), new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
        self.image(img_path, x=(PAGE_W - img_w) / 2, w=img_w, h=img_h)
        self.ln(img_h + 6)


# ── Load data ──
sections = load_json(SECTIONS_FILE)
diagram_meta = load_json(DIAGRAM_META_FILE)
all_diagrams = sorted(glob(f"{DIAGRAMS_DIR}/diagram_*.png"))

# Build lookup: section_index -> diagrams to show after it
diagrams_after = {}
for dm in diagram_meta:
    diagrams_after.setdefault(dm.get("after_section", 0), []).append(dm)

# Fallback: distribute evenly if no metadata
if not diagram_meta and all_diagrams:
    for i, path in enumerate(all_diagrams):
        sec_idx = min(i, len(sections) - 1) if sections else 0
        diagrams_after.setdefault(sec_idx + 1, []).append({"path": path, "label": f"Diagram {i+1}"})

# ── Build PDF ──
pdf = ResearchPDF()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=MARGIN)

# Title page
pdf.add_page()
pdf.ln(60)
pdf.set_font("Helvetica", "B", 24)
pdf.set_text_color(30, 30, 30)
pdf.multi_cell(0, 12, sanitize(user_topic), align="C")
pdf.ln(10)
pdf.set_font("Helvetica", "", 14)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 10, "A Visual Research Report", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "I", 11)
pdf.cell(0, 10, "Generated by an AI Agent using LangChain + Mermaid", align="C", new_x="LMARGIN", new_y="NEXT")

# Sections with interleaved diagrams
for i, sec in enumerate(sections):
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 16)
    pdf.set_text_color(30, 30, 30)
    pdf.multi_cell(0, 10, sanitize(f"{i + 1}. {sec['title']}"))
    pdf.ln(4)
    pdf.write_markdown(sec["content"])
    pdf.ln(6)
    for dm in diagrams_after.get(i + 1, []):
        if os.path.exists(dm["path"]):
            pdf.add_image_fit(dm["path"], dm["label"])

# Remaining diagrams not tied to any section
shown = {dm["path"] for dms in diagrams_after.values() for dm in dms}
remaining = [p for p in all_diagrams if p not in shown]
if remaining:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 18)
    pdf.set_text_color(30, 30, 30)
    pdf.cell(0, 14, "Additional Diagrams", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(6)
    for idx, path in enumerate(remaining):
        pdf.add_image_fit(path, f"Diagram {idx + 1}")

# Save
pdf_path = f"{OUTPUT_DIR}/Research_Report.pdf"
pdf.output(pdf_path)
log(f"PDF       -> Saved {pdf_path} ({len(sections)} sections, {len(all_diagrams)} diagrams)")